# Nasri Subtitle Studio — إضافة ترجمة SRT إلى MKV

أداة تعمل داخل Google Colab: تدخل رابط تنزيل مباشر للفيلم، وترفع ملف SRT، ثم تُنتج MKV مع مسار ترجمة قابل للتشغيل والإيقاف. لا تحرق الترجمة على الصورة، وتنسخ مسارات الفيديو والصوت دون إعادة ترميز.

**ملاحظات:** استخدم فقط روابط تملك حق الوصول إليها. يجب أن يكون الرابط رابط ملف مباشرًا، لا صفحة مشاهدة. رابط Gradio مؤقت، ويعمل ما دامت جلسة Colab نشطة. لا تغلق الجلسة قبل انتهاء المعالجة وتنزيل الناتج.

In [ ]:
!pip -q install -U gradio requests
!ffmpeg -version | head -n 2


In [ ]:
import os, re, time, shutil, subprocess, tempfile, threading
from pathlib import Path
from urllib.parse import urlparse, unquote

import requests
import gradio as gr

WORKDIR = Path("/content/nasri_subtitle_studio")
WORKDIR.mkdir(parents=True, exist_ok=True)
MAX_URL_LENGTH = 4096
CHUNK_SIZE = 1024 * 1024  # 1 MiB

def safe_filename(name, fallback="movie"):
    name = unquote(name or "").strip()
    name = re.sub(r"[\\/:*?\"<>|\x00-\x1f]", "_", name)
    name = re.sub(r"\s+", " ", name).strip(" .")
    if not name:
        name = fallback
    return name[:120]

def validate_direct_url(url):
    url = (url or "").strip()
    if not url:
        raise ValueError("أدخل رابط الفيلم المباشر.")
    if len(url) > MAX_URL_LENGTH:
        raise ValueError("الرابط طويل أكثر من اللازم.")
    parsed = urlparse(url)
    if parsed.scheme not in ("http", "https") or not parsed.netloc:
        raise ValueError("يجب أن يبدأ الرابط بـ http:// أو https://.")
    # Prevent accidental local/private-network URL targets.
    host = (parsed.hostname or "").lower()
    if host in ("localhost", "127.0.0.1", "::1") or host.endswith(".local"):
        raise ValueError("هذا العنوان محلي وغير مسموح.")
    return url

def download_movie(url, progress=gr.Progress()):
    url = validate_direct_url(url)
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124 Safari/537.36",
        "Accept": "*/*",
    })
    try:
        with session.get(url, stream=True, timeout=(20, 90), allow_redirects=True) as response:
            response.raise_for_status()
            content_type = response.headers.get("Content-Type", "").lower()
            if "text/html" in content_type:
                raise ValueError("الرابط أعاد صفحة HTML وليس ملف فيديو مباشرًا. استخدم رابط MP4/MKV مباشرًا.")
            length = response.headers.get("Content-Length")
            total = int(length) if length and length.isdigit() else None
            disposition = response.headers.get("Content-Disposition", "")
            match = re.search(r'filename\\*?=(?:UTF-8\\'\\')?["\\']?([^"\\';]+)', disposition, re.I)
            suggested = match.group(1).strip() if match else ""
            if not suggested:
                suggested = Path(urlparse(response.url).path).name
            name = safe_filename(suggested, "movie")
            suffix = Path(name).suffix.lower()
            if suffix not in (".mp4", ".mkv", ".mov", ".webm", ".m4v", ".avi", ".ts", ".mpg", ".mpeg"):
                # Do not trust extension alone; ffmpeg probes the contents.
                name = safe_filename(Path(name).stem or "movie") + ".video"
            dest = WORKDIR / ("source_" + name)
            downloaded = 0
            started = time.time()
            with open(dest, "wb") as f:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                    if not chunk:
                        continue
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        progress(min(downloaded / total, 0.99), desc=f"تنزيل الفيلم: {downloaded/1024/1024:.1f} / {total/1024/1024:.1f} MiB")
                    else:
                        progress(desc=f"تنزيل الفيلم: {downloaded/1024/1024:.1f} MiB")
            if downloaded < 1024:
                dest.unlink(missing_ok=True)
                raise ValueError("الملف الذي نُزّل صغير جدًا؛ ربما الرابط ليس رابط فيديو.")
            progress(1.0, desc=f"اكتمل تنزيل الفيلم ({downloaded/1024/1024:.1f} MiB)")
            return dest, name, downloaded
    except requests.exceptions.Timeout as e:
        raise RuntimeError("انتهت مهلة الاتصال أثناء التنزيل. جرّب الرابط مجددًا أو استخدم اتصالًا أكثر استقرارًا.") from e
    except requests.exceptions.HTTPError as e:
        code = e.response.status_code if e.response is not None else "غير معروف"
        raise RuntimeError(f"رفض الخادم التنزيل (HTTP {code}). قد يحتاج الرابط إلى تسجيل دخول أو انتهت صلاحيته.") from e
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"فشل الاتصال بالرابط: {e}") from e
    finally:
        session.close()

def probe_video(path):
    cmd = [
        "ffprobe", "-v", "error", "-show_entries",
        "format=duration,size,format_name:stream=codec_type,codec_name,width,height",
        "-of", "json", str(path)
    ]
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=90)
    if p.returncode != 0:
        raise RuntimeError("تعذّر على FFmpeg قراءة الملف. قد يكون ناقصًا أو غير مدعوم.\n" + p.stderr[-1500:])
    import json
    info = json.loads(p.stdout)
    streams = info.get("streams", [])
    if not any(s.get("codec_type") == "video" for s in streams):
        raise RuntimeError("الملف لا يحتوي على مسار فيديو قابل للقراءة.")
    fmt = info.get("format", {})
    return info, fmt

def mux_subtitle(url, srt_file, output_name, progress=gr.Progress()):
    # Clean old temporary files from previous runs.
    for p in WORKDIR.glob("source_*"):
        try: p.unlink()
        except: pass
    for p in WORKDIR.glob("output_*.mkv"):
        try: p.unlink()
        except: pass

    if srt_file is None:
        raise ValueError("ارفع ملف الترجمة SRT.")
    srt_path = Path(srt_file)
    if not srt_path.exists() or srt_path.suffix.lower() != ".srt":
        raise ValueError("الملف المرفوع يجب أن يكون بامتداد .srt.")
    # Verify text can be read; SRT should be UTF-8. Try UTF-8 BOM then UTF-8.
    raw = srt_path.read_bytes()
    try:
        subtitle_text = raw.decode("utf-8-sig")
    except UnicodeDecodeError as e:
        raise ValueError("ملف SRT ليس بترميز UTF-8. احفظه بترميز UTF-8 ثم ارفعه مجددًا.") from e
    if "-->" not in subtitle_text or not re.search(r"\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->", subtitle_text):
        raise ValueError("لم نعثر على توقيتات SRT المعتادة داخل الملف.")
    clean_name = safe_filename(output_name, "Nasri_Subtitle_Studio")
    clean_name = re.sub(r"(?i)\.mkv$", "", clean_name) + ".mkv"

    yield "⏬ بدء تنزيل الفيلم من الرابط المباشر…", None
    movie_path, original_name, size = download_movie(url, progress)
    yield f"✅ اكتمل التنزيل: {size/1024/1024:.1f} MiB\n🔎 فحص الفيديو والصوت…", None

    try:
        info, fmt = probe_video(movie_path)
    except Exception:
        movie_path.unlink(missing_ok=True)
        raise

    streams = info.get("streams", [])
    video_streams = [s for s in streams if s.get("codec_type") == "video"]
    audio_count = sum(1 for s in streams if s.get("codec_type") == "audio")
    duration = fmt.get("duration")
    dur_text = f"{float(duration)/60:.1f} دقيقة" if duration else "غير معروفة المدة"
    video_codec = video_streams[0].get("codec_name", "غير معروف")
    width = video_streams[0].get("width", "?")
    height = video_streams[0].get("height", "?")
    yield f"🎞️ الفيديو: {width}×{height}، الترميز {video_codec}، المدة {dur_text}\n🔊 المسارات الصوتية: {audio_count}\n🧩 إضافة الترجمة دون حرقها…", None

    out_path = WORKDIR / ("output_" + clean_name)
    # Copy all original video/audio/subtitle streams; append the new SRT track.
    # No video/audio re-encoding. Matroska supports SRT subtitles natively.
    cmd = [
        "ffmpeg", "-hide_banner", "-y", "-nostdin",
        "-i", str(movie_path), "-i", str(srt_path),
        "-map", "0:v?", "-map", "0:a?", "-map", "0:s?", "-map", "1:0",
        "-map_metadata", "0", "-map_chapters", "0",
        "-c", "copy",
        "-metadata:s:s", "language=ara",
        "-metadata:s:s", "title=Arabic",
        "-disposition:s", "0",
        "-progress", "pipe:1", "-nostats",
        str(out_path)
    ]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
    duration_seconds = float(duration) if duration else None
    last_status = "⚙️ جارٍ إنشاء ملف MKV…"
    try:
        while True:
            line = process.stdout.readline()
            if line == "" and process.poll() is not None:
                break
            line = line.strip()
            if line.startswith("out_time_ms=") and duration_seconds:
                try:
                    micros = int(line.split("=",1)[1])
                    frac = min(micros / (duration_seconds * 1_000_000), 0.99)
                    progress(frac, desc=f"دمج الترجمة: {frac*100:.0f}%")
                    last_status = f"⚙️ جارٍ دمج الترجمة: {frac*100:.0f}%"
                except ValueError:
                    pass
            elif line == "progress=end":
                progress(1.0, desc="اكتمل دمج الترجمة")
        stderr = process.stderr.read() if process.stderr else ""
        code = process.wait()
        if code != 0:
            out_path.unlink(missing_ok=True)
            raise RuntimeError("فشل FFmpeg في إنشاء MKV:\n" + stderr[-3000:])
        if not out_path.exists() or out_path.stat().st_size < 1024:
            raise RuntimeError("انتهت العملية لكن ملف الناتج غير موجود أو فارغ.")
    finally:
        if process.poll() is None:
            process.kill()
        if process.stdout: process.stdout.close()
        if process.stderr: process.stderr.close()

    final_size = out_path.stat().st_size
    # Remove the downloaded source to save Colab disk space; keep the output.
    movie_path.unlink(missing_ok=True)
    yield (
        f"✅ اكتملت العملية بنجاح!\n\n"
        f"📦 الملف: {out_path.name}\n"
        f"📏 الحجم النهائي: {final_size/1024/1024:.1f} MiB\n"
        f"🎬 الفيديو والصوت: نسخ دون إعادة ترميز\n"
        f"💬 الترجمة: مسار عربي قابل للاختيار، غير محروق على الصورة\n"
        f"📁 الصيغة: MKV",
        str(out_path)
    )

with gr.Blocks(title="Nasri Subtitle Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🎬 Nasri Subtitle Studio\n"
        "### إضافة ترجمة SRT كمسار مستقل إلى فيلم MKV — دون حرق الترجمة\n"
        "ألصق رابط تنزيل مباشر للفيلم وارفع ملف SRT بترميز UTF-8. تتم معالجة الملف داخل جلسة Colab."
    )
    with gr.Column():
        url_input = gr.Textbox(
            label="رابط الفيلم المباشر",
            placeholder="https://example.com/movie.mp4",
            lines=2
        )
        srt_input = gr.File(label="ملف الترجمة SRT (UTF-8)", file_types=[".srt"], type="filepath")
        name_input = gr.Textbox(label="اسم الملف الناتج (اختياري)", value="Nasri_Subtitle_Studio")
        run_btn = gr.Button("🚀 تنزيل الفيلم وإضافة الترجمة", variant="primary")
        status = gr.Textbox(label="حالة العملية", lines=8, interactive=False)
        output_file = gr.File(label="تنزيل ملف MKV الناتج")
    run_btn.click(
        fn=mux_subtitle,
        inputs=[url_input, srt_input, name_input],
        outputs=[status, output_file],
        show_progress="full"
    )
    gr.Markdown(
        "**ملاحظات مهمة:**\n"
        "- يجب أن يكون الرابط مباشرًا ويمكن تنزيله دون تسجيل دخول أو ملفات تعريف ارتباط خاصة.\n"
        "- قد لا تعمل الروابط المؤقتة أو روابط صفحات المشاهدة.\n"
        "- لا تُغلق جلسة Colab حتى تنتهي المعالجة وتنزّل الملف.\n"
        "- تأكد من وجود مساحة كافية في Colab؛ الملف الناتج يحتاج مساحة إضافية.\n"
        "- استخدم الأداة فقط مع ملفات لديك الحق في تنزيلها ومعالجتها."
    )

demo.queue(max_size=3)
demo.launch(share=True, debug=False)